# Tutorial 03: Automated Reporting

Wintermute automates the generation of professional security assessment reports. It uses a template-based approach where a main document is rendered, and specific findings (vulnerabilities or test runs) are appended using specialized templates.

## 1. Configuring the Reporting Backend

The `DocxTplPerVulnBackend` uses `docxtpl` for Jinja2-style templating in Word and `docxcompose` to merge multiple documents.

In [ ]:
from wintermute.backends.docx_reports import DocxTplPerVulnBackend
from wintermute.reports import Report, ReportSpec

# Register the DOCX backend pointing to our professional templates
Report.register_backend(
    "word_tpl_per_vuln",
    DocxTplPerVulnBackend(
        template_dir="templates",
        main_template="report_main.docx",
        vuln_template="report_vuln.docx",
    ),
    make_default=True,
)

print("Professional DOCX reporting backend registered.")

## 2. Defining Findings

Reports are generated from objects that contain vulnerabilities. Wintermute can automatically collect vulnerabilities from a hierarchy of `BaseModel` objects like `CloudAccount` or `Peripheral`.

In [ ]:
from wintermute.basemodels import Peripheral
from wintermute.findings import ReproductionStep, Risk, Vulnerability

# A hardware peripheral with a high-impact vulnerability
periph = Peripheral(
    name="SoC-UART-0",
    device_path="/dev/ttyUSB0",
    pType="UART",
    vulnerabilities=[
        Vulnerability(
            title="U-Boot Environment Variable Injection",
            description="The U-Boot bootloader allows arbitrary modification of environment variables via the UART console. An attacker can inject 'init=/bin/sh' into the 'bootargs' variable to gain a root shell on the underlying Linux system.",
            risk=Risk(likelihood="High", impact="Critical", severity="Critical"),
            reproduction_steps=[
                ReproductionStep(
                    title="Inject init=/bin/sh",
                    description="Modify bootargs to bypass init and launch a shell.",
                    tool="uboot-write-env",
                    action="setenv",
                    arguments=[
                        "bootargs",
                        "console=ttyS0,115200n8 root=/dev/mmcblk0p2 rw init=/bin/sh",
                    ],
                )
            ],
            verified=True,
        ),
        Vulnerability(
            title="Unauthenticated UART Root Shell",
            description="The UART console provides an unauthenticated root shell upon boot completion, allowing full system compromise via physical access.",
            cvss=9.0,
            risk=Risk(likelihood="High", impact="Critical", severity="Critical"),
            verified=True,
        ),
    ],
)

# Another peripheral illustrating flash read protection bypass
spi_flash = Peripheral(
    name="SPI-Flash-Main",
    device_path="SPI0",
    pType="SPI",
    vulnerabilities=[
        Vulnerability(
            title="SPI Flash Read Protection Bypass",
            description="The SPI flash chip lacks read protection, allowing the extraction of the entire firmware image using a hardware programmer.",
            cvss=7.5,
            reproduction_steps=[
                ReproductionStep(
                    title="Dump Firmware",
                    tool="flashrom",
                    action="read",
                    arguments=[
                        "-p",
                        "buspirate_spi:dev=/dev/ttyUSB0",
                        "-r",
                        "firmware_dump.bin",
                    ],
                )
            ],
            verified=True,
        )
    ],
)

print("Hardware findings prepared for professional reporting.")

## 3. Generating the Report

Finally, we define a `ReportSpec` and call `Report.save`. The system will crawl the provided objects, collect all vulnerabilities, and render the final DOCX file.

In [ ]:
spec = ReportSpec(
    title="Hardware Security Audit – Project Wintermute",
    author="Lead Security Researcher",
    summary="The assessment identified critical vulnerabilities in the bootloader and physical interface security. Mitigation requires disabling the U-Boot console and enforcing SPI flash read protection.",
)

# Save the report to disk
Report.save(spec, [periph, spi_flash], "Hardware_Security_Audit.docx")

print("Report 'Hardware_Security_Audit.docx' generated successfully.")